# 06 — Modelado multi-clase por familias

**Proyecto:** VIGÍA (Sentry) — TFC
**Objetivo:** entrenar un detector multi-clase sobre las 7 familias operativas de ataque (BENIGN + DoS + Probe + DDoS + Brute-force + Web + Bot).

**Entradas:** los mismos Parquet del split (train/val/test) generados en el 04.

**Decisiones que se toman:**
- Target multi-clase: `target_family` (8 valores, incluido "Otros").
- Modelo principal: XGBoost con `objective="multi:softprob"`.
- Balanceo: `sample_weight` por clase (equivalente multi-clase de scale_pos_weight).
- Métricas: F1 macro, F1 por clase, matriz de confusión.

**Motivo de seguridad:** este modelo es el que un analista de SOC usaría realmente. La granularidad por familia permite priorizar la respuesta y elegir contramedidas específicas.

In [ ]:
# ============================================
# Setup notebook 06
# ============================================
import os
from google.colab import drive
from pyspark.sql import SparkSession

GITHUB_USER = "AguCS231"
REPO_NAME   = "sentry"
REPO_LOCAL  = f"/content/{REPO_NAME}"
BASE_SPLITS = "/content/drive/MyDrive/cic-ids2017-splits"

if not os.path.exists(REPO_LOCAL):
    %cd /content
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_LOCAL}
    !git config --global user.email "srcasares2@gmail.com"
    !git config --global user.name "{GITHUB_USER}"
else:
    %cd {REPO_LOCAL}

drive.mount('/content/drive')

spark = SparkSession.builder \
    .appName("VIGIA_Familias") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"✅ Spark {spark.version}")

## 1. Carga de splits y definición de features

Usamos el mismo conjunto de features que el modelo B (sin `Destination_Port`) para mantener la comparabilidad.

**Motivo de seguridad:** coherencia metodológica. Si en binario decidimos que el puerto era un atajo, en familias también lo excluimos.

In [ ]:
# ============================================
# Cargar splits y definir features
# ============================================
from pyspark.sql.types import NumericType

train_df = spark.read.parquet(f"{BASE_SPLITS}/train.parquet")
val_df   = spark.read.parquet(f"{BASE_SPLITS}/val.parquet")
test_df  = spark.read.parquet(f"{BASE_SPLITS}/test.parquet")

# Mismas features que el modelo B (sin Destination_Port)
excluir = {"Label", "target_bin", "target_family", "Destination_Port"}
feature_cols = [
    c for c in train_df.columns
    if c not in excluir and isinstance(train_df.schema[c].dataType, NumericType)
]

print(f"Features: {len(feature_cols)}")
print(f"Train: {train_df.count():,} | Val: {val_df.count():,} | Test: {test_df.count():,}")

# Familias objetivo
familias = sorted([r["target_family"]
                   for r in train_df.select("target_family").distinct().collect()])
print(f"\nFamilias: {familias}")

## 2. Codificación del target multi-clase

XGBoost necesita etiquetas numéricas, no strings. Mapeamos las 7 familias a índices 0-6 y guardamos el mapeo para interpretar los resultados después.

**Motivo de seguridad:** mantener el mapeo explícito permite traducir las predicciones del modelo a nombres legibles, requisito para el dashboard operativo.

In [ ]:
# ============================================
# Codificación del target + muestreo de train
# ============================================
from pyspark.sql.functions import col, when

# Mapeo: familia → índice numérico
familias = ['BENIGN', 'DoS', 'Probe', 'DDoS', 'Brute-force', 'Web', 'Bot']
mapeo = {fam: i for i, fam in enumerate(familias)}
print("Mapeo de familias:")
for fam, idx in mapeo.items():
    print(f"  {idx} → {fam}")

# Aplicar mapeo a train y val (test mantiene "Otros" pero no lo usaremos en este notebook todavía)
def aplicar_mapeo(df):
    expr = None
    for fam, idx in mapeo.items():
        cond = when(col("target_family") == fam, idx)
        expr = cond if expr is None else expr.when(col("target_family") == fam, idx)
    return df.withColumn("target_family_idx", expr.otherwise(-1))

train_df = aplicar_mapeo(train_df)
val_df   = aplicar_mapeo(val_df)

# Verificar
print("\nDistribución train:")
train_df.groupBy("target_family", "target_family_idx").count().orderBy("target_family_idx").show()

# Muestreo estratificado al 50% (misma decisión que en 05)
print("Muestreando train al 50% (estratificado)...")
parts = []
for fam in familias:
    parts.append(train_df.filter(col("target_family") == fam).sample(fraction=0.5, seed=42))

from functools import reduce
from pyspark.sql import DataFrame
train_sample = reduce(DataFrame.unionByName, parts)
train_sample.cache()

print(f"Train completo:   {train_df.count():>10,}")
print(f"Train muestreado: {train_sample.count():>10,}")

In [ ]:
# ============================================
# Conversión a numpy (train muestreado + val completo)
# ============================================
import numpy as np
import time, gc

print("Convirtiendo train muestreado...")
t0 = time.time()
train_pd = train_sample.select(feature_cols + ["target_family_idx"]).toPandas()
X_train = train_pd[feature_cols].values.astype(np.float32)
y_train = train_pd["target_family_idx"].values.astype(np.int8)
del train_pd; gc.collect()
print(f"✅ X_train: {X_train.shape}  ({X_train.nbytes/1e6:.0f} MB) en {time.time()-t0:.1f}s")

print("\nConvirtiendo val...")
t0 = time.time()
val_pd = val_df.select(feature_cols + ["target_family_idx"]).toPandas()
X_val = val_pd[feature_cols].values.astype(np.float32)
y_val = val_pd["target_family_idx"].values.astype(np.int8)
del val_pd; gc.collect()
print(f"✅ X_val: {X_val.shape} en {time.time()-t0:.1f}s")

# Verificar codificación
print(f"\nClases presentes en train: {sorted(np.unique(y_train))}")
print(f"Clases presentes en val:   {sorted(np.unique(y_val))}")

In [ ]:
# ============================================
# Pesos por clase (equivalente multi-clase de scale_pos_weight)
# ============================================
from sklearn.utils.class_weight import compute_sample_weight

# Estrategia: peso inversamente proporcional a la frecuencia
# La clase más frecuente (BENIGN) tiene peso 1, las minoritarias más
pesos = compute_sample_weight(class_weight="balanced", y=y_train)
print(f"Pesos calculados: min={pesos.min():.2f}, max={pesos.max():.2f}")

# Mostrar peso medio por clase
for idx, fam in enumerate(familias):
    mask = y_train == idx
    if mask.sum() > 0:
        print(f"  Clase {idx} ({fam:12s}): {mask.sum():>7,} muestras, peso medio {pesos[mask].mean():.2f}")

In [ ]:
# ============================================
# Entrenar XGBoost multi-clase por familias
# ============================================
import xgboost as xgb
import time

modelo_fam = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=7,
    eval_metric="mlogloss",
    max_depth=6,
    n_estimators=200,          # más que en binario: problema más complejo
    learning_rate=0.1,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

print("Entrenando modelo por familias (3-8 min)...")
t0 = time.time()
modelo_fam.fit(
    X_train, y_train,
    sample_weight=pesos,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
print(f"✅ Entrenado en {time.time()-t0:.1f}s")

## 3. Evaluación en validation

Además de las métricas globales, analizamos **F1 por clase** y la **matriz de confusión**.

**Motivo de seguridad:** el F1 macro puede esconder un mal rendimiento en clases minoritarias. Un IDS que detecta bien DoS pero falla en Web Attack es un IDS incompleto. Hay que mirar el desglose.

In [ ]:
# ============================================
# Evaluación multi-clase en validation
# ============================================
from sklearn.metrics import (
    f1_score, classification_report,
    confusion_matrix
)
import numpy as np

y_pred_fam = modelo_fam.predict(X_val)

# Métricas globales
f1_macro = f1_score(y_val, y_pred_fam, average="macro")
f1_weighted = f1_score(y_val, y_pred_fam, average="weighted")

print("=== Modelo por familias — Validation ===\n")
print(f"F1 macro:    {f1_macro:.4f}   (media sin ponderar por frecuencia)")
print(f"F1 weighted: {f1_weighted:.4f}   (ponderado por frecuencia)")

print("\n=== F1 y métricas por clase ===\n")
print(classification_report(
    y_val, y_pred_fam,
    labels=list(range(7)),
    target_names=familias,
    digits=4,
    zero_division=0,
))

print("=== Matriz de confusión ===\n")
cm = confusion_matrix(y_val, y_pred_fam)
print(f"{'':>12}", end="")
for f in familias:
    print(f"{f[:8]:>9}", end="")
print()
for i, fam in enumerate(familias):
    print(f"{fam:>12}", end="")
    for j in range(len(familias)):
        print(f"{cm[i,j]:>9,}", end="")
    print()

In [ ]:
# ============================================
# Guardar modelo por familias
# ============================================
import os, json

MODELS_DIR = "/content/sentry/src/models"
os.makedirs(MODELS_DIR, exist_ok=True)

modelo_fam.save_model(f"{MODELS_DIR}/familias_sin_puerto.json")

meta = {
    "fecha": "2026-10-01",
    "tipo": "multi-clase por familias (7 clases)",
    "features": feature_cols,
    "n_features": len(feature_cols),
    "mapeo_familias": mapeo,
    "n_train": int(len(y_train)),
    "n_val": int(len(y_val)),
    "muestreo_train": "50% estratificado (seed=42)",
    "hiperparametros": {
        "objective": "multi:softprob",
        "num_class": 7,
        "max_depth": 6,
        "n_estimators": 200,
        "learning_rate": 0.1,
        "random_state": 42,
    },
    "balanceo": "compute_sample_weight balanced",
    "metricas_val": {
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted),
    },
    "f1_por_clase": {
        fam: float(f1_score(y_val, y_pred_fam, labels=[i], average="macro", zero_division=0))
        for i, fam in enumerate(familias)
    },
}

with open(f"{MODELS_DIR}/familias_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print("✅ Modelo y metadatos guardados")
!ls -lh {MODELS_DIR}/

In [ ]:
%cd /content/sentry

from datetime import datetime
fecha = datetime.now().strftime("%Y-%m-%d")

entrada = f"""

---

## {fecha} — Modelo por familias (Fase 2, primera versión)

**Qué probé:**
- XGBoost multi-clase sobre 7 familias (BENIGN, DoS, Probe, DDoS, Brute-force, Web, Bot).
- Features: las mismas 77 del modelo binario B (sin Destination_Port).
- Muestreo de train al 50% (990.947 filas) por límites de RAM en Colab.
- Balanceo con `compute_sample_weight(class_weight="balanced")` → pesos entre 0.18 (BENIGN) y 207.27 (Bot).

**Resultados en validation:**
- F1 macro: 0.9575
- F1 weighted: 0.9988
- F1 por clase:
  - BENIGN: 0.9992 | DoS: 0.9980 | Probe: 0.9965
  - DDoS: 0.9994 | Brute-force: 0.9993 | Web: 0.9726 | Bot: 0.7378

**Análisis de la clase Bot:**
- Recall 0.9931 (detecta todos los ataques reales).
- Precision 0.5869 → 202 BENIGN clasificados como Bot (0.06% del total BENIGN).
- Interpretación: el peso 207× provoca sobredetección en dirección Bot. Operativamente es ruido tolerable, metodológicamente es un hallazgo.

**Decisiones:**
- No reajustamos los pesos: el resultado es defendible y el análisis de la clase Bot es un aporte.
- El modelo se guarda como `familias_sin_puerto.json` con metadatos completos.
- Se mantiene la coherencia metodológica con el binario (mismas 77 features).

**Limitaciones conocidas:**
- Train al 50% por RAM.
- Clase Bot con precisión baja por peso desproporcionado.
- Evaluación pendiente sobre test (aún no ejecutada).

**Próximos pasos:**
- Experimento con/sin Destination_Port también en familias, para completar la Fase 1.
- SHAP por clase (Fase 2).
"""

with open("docs/diario.md", "a") as f:
    f.write(entrada)

print("✅ Diario actualizado")

!git add .
!git commit -m "Modelo por familias: XGBoost multi-clase, F1 macro 0.9575"

In [ ]:
from getpass import getpass

TOKEN = getpass("token: ")
print(f"Token capturado: {len(TOKEN)} caracteres")

%cd /content/sentry
!git remote set-url origin https://{TOKEN}@github.com/AguCS231/sentry.git
!git remote get-url origin | grep -q "ghp_" && echo "✅ Token configurado"
!git push